# Conversation Search Test

Calls `search_conversation_findings` directly with its injected arguments, so no graph run is needed. Set `CID` and `USER_ID` to a conversation with several verified turns and the user id its turns were written under. Requires turns whose `conversation_findings` rows have an embedding.

In [1]:
import json
import os
import sys
import time

sys.path.insert(0, "..")

from app.config import GCP_PROJECT_ID, get_secret

os.environ["GOOGLE_API_KEY"] = get_secret("gemini-api-key", GCP_PROJECT_ID)

from app.orchestrator.tools import search_conversation_findings

In [2]:
CID = "fd17774b-b81c-4dc3-864b-78f1e7e7b738"
USER_ID = "gateway-test-user"
BASE = {"conversation_id": CID, "user_id": USER_ID}


async def run_search(**args):
    start = time.perf_counter()
    result = await search_conversation_findings.ainvoke({**BASE, **args})
    print(f"{time.perf_counter() - start:.1f}s")
    return result


def show_hits(hits):
    if isinstance(hits, str):
        print(hits)
        return
    for hit in hits:
        print(f"{hit['distance']:.4f} | {hit['answered_at'][:16]} | {hit['title'] or '-'} | {hit['question'][:70]}")

## Tags

In [3]:
tags = await run_search(mode="tags")
print("topics:", [(t["tag"], t["count"]) for t in tags["topics"]])
print("entities:", [(t["tag"], t["count"]) for t in tags["entities"]])

2.9s
topics: [('shap feature importance', 7), ('model interpretability', 4), ('feature engineering', 3), ('financial impact', 3), ('eda', 2), ('model evaluation', 2), ('beeswarm plot', 1), ('cross-validation', 1), ('cyclical order patterns', 1), ('dashboard navigation', 1), ('data splitting', 1), ('ensemble models', 1), ('exploratory data analysis', 1), ('heatmap', 1), ('long-tail sparsity', 1), ('methodology', 1), ('model training pipeline', 1), ('ndcg@5', 1), ('order volume', 1), ('preliminary eda notebook', 1), ('profit lift', 1), ('profit lift drivers', 1), ('reorder rate analysis', 1), ('user exploratory ratio', 1), ('waterfall chart', 1)]
entities: [('xgboost ranker', 7), ('full ensemble (all base models)', 4), ('avg days between product purchases', 3), ('banana', 2), ('preliminary_eda.ipynb', 2), ('shap', 2), ('feature_eda_screening.ipynb', 1), ('heuristic baseline', 1), ('lightgbm ranker + lightgbm classifier', 1), ('lightgbm ranker + xgboost classifier', 1), ('product_order_an

In [4]:
tags = await run_search(mode="tags", scope="user")
print("topics:", [(t["tag"], t["count"]) for t in tags["topics"]])
print("entities:", [(t["tag"], t["count"]) for t in tags["entities"]])

1.2s
topics: [('shap feature importance', 7), ('eda', 5), ('feature engineering', 5), ('financial impact', 5), ('model interpretability', 4), ('financial impact modeling', 2), ('long-tail sparsity', 2), ('model evaluation', 2), ('order timing', 2), ('product sparsity', 2), ('profit lift drivers', 2), ('recall@5 to profit translation', 2), ('reorder behavior', 2), ('beeswarm plot', 1), ('champion model', 1), ('charts', 1), ('conversion profit lift', 1), ('cross-validation', 1), ('cyclical order patterns', 1), ('dashboard navigation', 1), ('dashboard page summary', 1), ('data exploration', 1), ('data overview', 1), ('data pipeline', 1), ('data splitting', 1), ('dataform', 1), ('ensemble models', 1), ('exploratory data analysis', 1), ('feature importance', 1), ('heatmap', 1), ('medallion architecture', 1), ('methodology', 1), ('model training pipeline', 1), ('ndcg@5', 1), ('order patterns', 1), ('order volume', 1), ('orders', 1), ('preliminary eda notebook', 1), ('produce department', 1),

## Search

In [7]:
QUERY = "which features matter most to the model"

hits = await run_search(mode="search", query=QUERY, top_k=10)
show_hits(hits)

2.0s
0.2478 | 2026-10-29T19:34 | XGBoost Ranker top 10 SHAP features and methodology | Show me a chart that displays feature importance for the top 10 most i
0.2585 | 2026-10-29T19:42 | SHAP beeswarm: XGBoost Ranker on test split | Can you create a beeswarm plot to illustrate feature importance for th
0.2695 | 2026-10-29T19:33 | XGBoost Ranker top 10 SHAP features and methodology | Show me a chart that displays feature importance for the top 10 most i
0.2703 | 2026-10-29T19:11 | Model Evaluation page: NDCG@5 and feature importance | I am looking at the model evaluation page at the test split with ndcg5
0.2766 | 2026-10-29T19:16 | XGBoost Ranker's top feature explained and validated | I want to understand what the XGBoost Ranker model leans on most. Show
0.2833 | 2026-10-29T19:36 | SHAP waterfall: XGBoost Ranker prediction for Banana | Can you show me a waterfall chart the illustrates impact of each featu
0.3040 | 2026-10-29T19:38 | Conversation summary: model evaluation, SHAP, and fina

In [6]:
hits = await run_search(mode="search", query=QUERY, scope="user")
show_hits(hits)

2.0s
0.2478 | 2026-10-29T19:34 | XGBoost Ranker top 10 SHAP features and methodology | Show me a chart that displays feature importance for the top 10 most i
0.2585 | 2026-10-29T19:42 | SHAP beeswarm: XGBoost Ranker on test split | Can you create a beeswarm plot to illustrate feature importance for th
0.2695 | 2026-10-29T19:33 | XGBoost Ranker top 10 SHAP features and methodology | Show me a chart that displays feature importance for the top 10 most i
0.2703 | 2026-10-29T19:11 | Model Evaluation page: NDCG@5 and feature importance | I am looking at the model evaluation page at the test split with ndcg5
0.2766 | 2026-10-29T19:16 | XGBoost Ranker's top feature explained and validated | I want to understand what the XGBoost Ranker model leans on most. Show


## Filters and tool calls

In [10]:
TOPICS = []     # fill from the tags output above
ENTITIES = ['full ensemble (all base models)']

hits = await run_search(mode="search", query=QUERY, topics=TOPICS, entities=ENTITIES)
show_hits(hits)

1.9s
0.2703 | 2026-10-29T19:11 | Model Evaluation page: NDCG@5 and feature importance | I am looking at the model evaluation page at the test split with ndcg5
0.3040 | 2026-10-29T19:38 | Conversation summary: model evaluation, SHAP, and financial impact | Can you summarize the key findings from this conversation?
0.3312 | 2026-10-29T19:31 | Ensemble profit lift by driver, test split | Show me a stacked bar chart of each ensemble model's total annual prof
0.3405 | 2026-10-29T19:45 | Financial Impact page overview and navigation | What does this page show? How do I navigate it?


In [11]:
hits = await run_search(mode="search", query=QUERY, top_k=1, include_tool_calls=True)
print(json.dumps(hits[0]["tool_calls"], indent=1)[:1500])

1.6s
[
 {
  "name": "run_dax_query",
  "args": {
   "args": {
    "dax": "EVALUATE TOPN(10, FILTER(power_bi_shap_barplots, power_bi_shap_barplots[model_name]=\"XGBoost Ranker\" && power_bi_shap_barplots[split]=\"test\"), power_bi_shap_barplots[mean_absolute_shap], DESC)"
   }
  },
  "ref_id": "ref_1"
 },
 {
  "name": "get_repo_contents",
  "args": {
   "args": {
    "path": "Notebooks/evaluation"
   }
  },
  "ref_id": null
 },
 {
  "name": "search_docs",
  "args": {
   "query": "SHAP mean absolute importance conclusions which features drive reorder predictions most",
   "top_k": 8
  },
  "ref_id": null
 },
 {
  "name": "get_repo_contents",
  "args": {
   "args": {
    "path": "Notebooks/evaluation/model_evaluation.ipynb"
   }
  },
  "ref_id": null
 },
 {
  "name": "generate_chart",
  "args": {
   "source_ref": "ref_1",
   "spec": {
    "chart_type": "horizontal_bar",
    "sort_order": "asc",
    "title": "Top 10 SHAP Features: XGBoost Ranker (Test)",
    "x_field": "power_bi_shap_barpl

In [12]:
print(sorted(hits[0]))

['active_page', 'answer_markdown', 'answered_at', 'conversation_id', 'distance', 'entities', 'filter_context', 'question', 'sources', 'title', 'tool_calls', 'topics', 'verified_numbers']


## Argument errors

In [13]:
print(await run_search(mode="search"))
print(await run_search(mode="search", query=QUERY, top_k=11))
print(await run_search(mode="tags", query=QUERY))

0.0s
1 validation error for SearchConversationFindingsArgs
  Value error, mode='search' needs a query. [type=value_error, input_value={'conversation_id': 'fd17...user', 'mode': 'search'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
0.0s
1 validation error for SearchConversationFindingsArgs
  Value error, top_k must be between 1 and 10. [type=value_error, input_value={'conversation_id': 'fd17...the model', 'top_k': 11}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
0.0s
1 validation error for SearchConversationFindingsArgs
  Value error, mode='tags' takes only scope. Use mode='search' to search. [type=value_error, input_value={'conversation_id': 'fd17...tter most to the model'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


## Overview with summaries

In [5]:
from app.orchestrator.tools import get_conversation_findings

overview = await get_conversation_findings.ainvoke({"view": "overview", "conversation_id": CID})
print({k: v for k, v in overview.items() if k not in ("summaries", "turns")})
for summary in overview["summaries"]:
    print(f"\nturns {summary['first_turn']}-{summary['last_turn']} | {summary['title']} | {summary['topics']}")
    print(summary["summary"][:500])
print()
for turn in overview["turns"][:30]:
    print(turn["turn"], turn["title"], "|", turn["question"][:60])

{'total_turns': 24, 'total_approx_tokens': 18007, 'token_budget': 20000, 'max_recent_turns_within_budget': 24, 'first_answered_at': '2026-10-29T19:09:03.600343+00:00', 'last_answered_at': '2026-10-30T02:06:41.532637+00:00'}

turns 1-9 | Dashboard model evaluation, SHAP, and financial impact | ['model evaluation', 'NDCG@5', 'SHAP feature importance', 'feature engineering validation', 'EDA replication', 'train/test split code', 'StratifiedGroupKFold', 'financial impact by driver', 'SHAP waterfall', 'Banana product explanation']
The user first asked for a page summary but was unclear which page they were on; they then confirmed the Model Evaluation page (test split, NDCG@5, bar plot view). On NDCG@5, the Full Ensemble (champion, chosen on Recall@5) scores about 0.536, essentially tied with the best single base models; all base models and ensembles cluster tightly, while the heuristic baseline (prior cart position) sits well below at about 0.378, a 41.68% relative improvement.

SHAP featur